# 模块三：神经网络积木库

模块二带你理解了梯度的底层流动逻辑，而模块三的目标是教你如何将这些逻辑封装成可复用的“功能积木”，搭建出能够解决实际问题（如 MNIST 识别）的神经网络。

## 单元 3.1：nn.Module 类体系 —— 模型的“骨架”

在 PyTorch 中，无论是简单的线性层（Linear），还是复杂的卷积神经网络（CNN），所有的网络组件都必须继承自 `nn.Module`。它是所有神经网络积木的基类（Base Class）。

### 1. 核心定义与机制

在开始写代码前，我们需要掌握几个面向对象编程（OOP）在 PyTorch 中的特有应用：

*   **继承 (Inheritance)**：子类（如你定义的模型）获得父类（`nn.Module`）的所有属性和方法。这让你无需从零实现复杂的梯度追踪管理。
*   **参数登记 (Parameter Registration)**：当你把一个 PyTorch 的层（如 `nn.Linear`）赋值为类的属性（`self.xxx = ...`）时，`nn.Module` 内部会利用 Python 的魔术方法自动捕获这个对象，并将其包含的权重（Weight）和偏置（Bias）登记在册。
*   **状态转换 (State Management)**：`nn.Module` 提供了一键切换模型状态的功能，例如 `.to(device)` 移动到 GPU，或 `.eval()` 切换到推理模式。

### 2. 构建模型的标准模板

请在 Jupyter Notebook 中运行以下代码，观察一个标准 PyTorch 模型的构造：

In [2]:
import torch
import torch.nn as nn

# 定义一个继承自 nn.Module 的类
class SingleNeuron(nn.Module):
    def __init__(self, input_dim, output_dim):
        """
        __init__: 构造函数，用于初始化积木（层）
        作用：在这里定义模型中包含的所有可学习参数的层。
        """
        # super().__init__(): 调用父类 nn.Module 的初始化方法
        # 作用：这是必须的一步！它会启动 nn.Module 的参数追踪机制。
        super().__init__()
        
        # nn.Linear(in, out): 线性层（全连接层）
        # 作用：执行 Y = XW^T + b 的运算。
        # 它会自动创建形状为 (output_dim, input_dim) 的权重和 (output_dim,) 的偏置。
        self.fc = nn.Linear(input_dim, output_dim)
        
    def forward(self, x):
        """
        forward: 前向传播函数
        作用：定义数据 x 经过积木的顺序和逻辑。
        """
        # 当你执行 model(x) 时，实际上就是在运行这个函数
        return self.fc(x)

# 实例化一个 784 维输入、10 维输出的模型（模拟 MNIST）
model = SingleNeuron(784, 10)

# 打印模型
print(f"模型结构:\n{model}")

模型结构:
SingleNeuron(
  (fc): Linear(in_features=784, out_features=10, bias=True)
)


### 3. 关键函数详解

*   **`model.named_parameters()`**：
    *   **作用**：迭代器，返回模型中所有已登记的参数名称及其对应的张量。
    *   **重要性**：在光子计算竞赛中，我们需要通过它提取特定层的权重矩阵，进行 8bit 量化或映射。
*   **`model.children()`**：
    *   **作用**：返回模型中定义的顶层子模块（積木）。

### 4. 深度解析：为什么不直接调用 `.forward()`？

在 Python 中，如果一个类实现了 `__call__` 方法，它的实例就可以像函数一样被调用。

当你执行 `output = model(input_data)` 时：
1.  PyTorch 内部调用了 `__call__`。
2.  它会先执行预定义的 **Hooks (钩子)**（用于性能监控、中间层特征提取或你以后要做的算力审计）。
3.  最后才调用你写的 `forward` 函数。
4.  **结论**：**永远不要写 `model.forward(x)`**，这会跳过所有重要的中间管理环节。

---

### 🚀 单元 3.1 练习任务

请在 Jupyter Notebook 中完成：

1.  **创建模型**：定义一个名为 `DoubleLinear` 的类。
2.  **定义层**：在 `__init__` 中定义两个线性层：
    *   `fc1`: 输入 784 维，输出 256 维。
    *   `fc2`: 输入 256 维，输出 10 维。
3.  **串联逻辑**：在 `forward` 中，让输入 `x` 先经过 `fc1`，得到的结果直接喂给 `fc2` 并返回。
4.  **算力审计初探**：实例化该模型，利用 `model.named_parameters()` 遍历所有参数，并编写一段代码计算：**该模型总共有多少个浮点数参数（权重元素 + 偏置元素之和）？**

**提示**：一个 `Linear(in_f, out_f)` 层的权重数量是 $in\_f \times out\_f$，偏置数量是 $out\_f$。

**完成后请给出你的代码和最终计算出的总参数量，我们将推进到 3.2 线性层（Linear）的深入剖析。**

In [5]:
import torch
import torch.nn as nn

class DoubleLinear(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        return self.fc2(self.fc1(x))
    
model = DoubleLinear(784, 128, 10)
print(f"模型结构:\n{model}")
# print(f"模型参数:\n{list(model.named_parameters())}")

模型结构:
DoubleLinear(
  (fc1): Linear(in_features=784, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=10, bias=True)
)



你的代码实现得非常标准！`DoubleLinear` 的架构逻辑完全正确。

关于**任务 4（计算总参数量）**，这是进行模型算力审计（Audit）的第一步。在光子计算中，每一个参数都对应着硬件上的一个配置数值，因此准确计算参数规模至关重要。

#### 任务 4 解决方案：参数量统计

你可以通过遍历 `parameters()` 并使用 `numel()` 函数来实现：

In [ ]:
# 计算总参数量
total_params = 0
for name, param in model.named_parameters():
    # numel(): Number of Elements
    # 作用：返回一个张量（Tensor）中包含的元素总数。
    count = param.numel()
    print(f"参数: {name} | 元素数量: {count}")
    total_params += count

print(f"\n该模型总参数量为: {total_params}")

# 学霸级数学验证：
# fc1.weight: 784 * 128 = 100,352
# fc1.bias: 128
# fc2.weight: 128 * 10 = 1,280
# fc2.bias: 10
# 总计: 100,352 + 128 + 1,280 + 10 = 101,770